# Working with HypEx Dataset

Build a small experiment table from user assignments and outcomes. Declare
what the columns mean, then use the same HypEx operations on pandas and
Spark. The worked example uses Python dictionaries as inputs and calls
HypEx for selection, filtering, joins, aggregates and reporting.

- [Build datasets and declare roles](#dataset-roles)
- [Select, join and inspect rows](#dataset-operations)
- [Summarize the experiment table](#dataset-summary)
- [Run the same workflow on Spark](#dataset-spark)

<a id="dataset-roles"></a>

## 1. Build datasets and declare roles

Each table has one row per user. `InfoRole` marks the join key and descriptive
fields, `TreatmentRole` identifies the assigned group, and `TargetRole`
marks an outcome. A role describes the purpose of a column. You can supply a
type as well, such as `TargetRole(float)`.

This example uses the inline `{"data": {...}}` format. With an existing
pandas or Spark table, pass that table as `data`. For Spark, supply the
session and select `backend=BackendsEnum.spark`. A CSV path also works as a
local input to `Dataset`.

In [1]:
import os

# Suppress the pandas-on-Spark timezone warning before importing HypEx.
os.environ.setdefault("PYARROW_IGNORE_TIMEZONE", "1")

from hypex.dataset import Dataset, InfoRole, SmallDataset, TargetRole, TreatmentRole
from hypex.utils import BackendsEnum

assignments = Dataset(
    roles={
        "user_id": InfoRole(int),
        "variant": TreatmentRole(str),
        "segment": InfoRole(str),
    },
    data={
        "data": {
            "user_id": [101, 102, 103, 104, 105, 106],
            "variant": ["control", "test", "control", "test", "control", "test"],
            "segment": ["new", "new", "returning", "returning", "new", "returning"],
        }
    },
)
outcomes = Dataset(
    roles={"user_id": InfoRole(int), "revenue": TargetRole(float)},
    data={
        "data": {
            "user_id": [101, 102, 103, 104, 105],
            "revenue": [10.0, 18.0, 0.0, 24.0, 8.0],
        }
    },
)

display(assignments)
print("Backend:", assignments.backend_type.value)
print("Outcome columns:", outcomes.search_columns(TargetRole(), search_types=[float]))

,user_id,variant,segment
0,101,control,new
1,102,test,new
2,103,control,returning
3,104,test,returning
4,105,control,new
5,106,test,returning


Backend: pandas
Outcome columns: ['revenue']


<a id="dataset-operations"></a>

## 2. Select, join and inspect rows

Start from assignments and left-join outcomes by user ID to keep users whose
outcome is missing. `merge` carries the roles into the result. In this
example, missing revenue means a record is unavailable, while zero is a
recorded outcome. Retain that distinction while checking the input.

Use `select` to project columns and a Boolean condition to filter rows. The
filtered result is also a `Dataset`.

In [2]:
cohort = assignments.merge(outcomes, on="user_id", how="left")
print(f"Users kept after the join: {len(cohort)} of {len(assignments)}")
print("Missing revenue records:", cohort.count_nulls()["revenue"])
print("Targets after the join:", cohort.search_columns(TargetRole()))

test_users = cohort[cohort["variant"] == "test"]
test_users.select(["user_id", "revenue"]).sort("user_id")

Users kept after the join: 6 of 6
Missing revenue records: 1
Targets after the join: ['revenue']


,user_id,revenue
1,102,18.0
3,104,24.0
5,106,NaN


<a id="dataset-summary"></a>

## 3. Summarize the experiment table

`value_counts` counts distinct combinations in the selected columns. Use
it to inspect assignment by segment. `groupby().agg()` gives one row per
variant. Here, mean revenue summarizes recorded outcomes, so check
coverage before using it as an experiment metric.

`SmallDataset` is the pandas-backed container for small report tables and
can be built from column lists or records using `SmallDataset.from_dict`.
Use `Dataset` for experiment input that may need a distributed backend.

In [3]:
def describe_cohort(data):
    allocation = data.select(["variant", "segment"]).value_counts()
    mean_revenue = data.groupby("variant").agg({"revenue": "mean"})
    return allocation, mean_revenue


allocation, mean_revenue = describe_cohort(cohort)
display(allocation, mean_revenue)

quality_report = SmallDataset.from_dict(
    {
        "assigned_users": [len(cohort)],
        "missing_revenue": [cohort.count_nulls()["revenue"]],
    },
    roles=InfoRole(int),
)
quality_report

,variant,segment,count
0,control,new,2
1,test,returning,2
2,control,returning,1
3,test,new,1


,revenue
variant,
control,6.0
test,21.0


,assigned_users,missing_revenue
0,6,1


<a id="dataset-spark"></a>

## 4. Run the same workflow on Spark

An active Spark session is needed to use the Spark backend. This cell starts
one locally for the tutorial and stops it at the end. On a cluster, use the
session provided by your environment.

`to_backend` transfers the tiny example tables to Spark and preserves their
roles. The same join and the same `describe_cohort` function run on the
distributed datasets. Compare the printed coverage and the group means with the pandas output above.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory

from pyspark.sql import SparkSession

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

with TemporaryDirectory(prefix="hypex-dataset-tutorial-") as spark_files:
    # Keep routine JVM warnings out of this local example. Errors remain visible.
    log_config = Path(spark_files) / "log4j2.properties"
    log_config.write_text(
        "rootLogger.level = error\n"
        "rootLogger.appenderRef.console.ref = console\n"
        "appender.console.type = Console\n"
        "appender.console.name = console\n"
        "appender.console.layout.type = PatternLayout\n"
        "appender.console.layout.pattern = %p %c: %m%n\n",
        encoding="utf-8",
    )
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("HypEx Dataset tutorial")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.shuffle.partitions", "2")
        .config(
            "spark.driver.extraJavaOptions",
            f"-Dlog4j.configurationFile={log_config.as_uri()}",
        )
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    try:
        spark_assignments = assignments.to_backend(BackendsEnum.spark, session=spark)
        spark_outcomes = outcomes.to_backend(BackendsEnum.spark, session=spark)
        spark_cohort = spark_assignments.merge(spark_outcomes, on="user_id", how="left")
        spark_allocation, spark_mean = describe_cohort(spark_cohort)

        print("Backend:", spark_cohort.backend_type.value)
        print("Missing revenue records:", spark_cohort.count_nulls()["revenue"])
        display(spark_allocation, spark_mean)
    finally:
        spark.stop()

Backend: spark
Missing revenue records: 1


,variant,segment,count
0,test,returning,2
1,control,new,2
2,test,new,1
3,control,returning,1


,revenue
variant,
control,6.0
test,21.0


A `Dataset` is the table you pass to HypEx. A custom pipeline uses
`ExperimentData` to carry that table together with intermediate results.
See [Custom pipelines](../advanced_tutorials/04_Custom_Pipelines.ipynb) for
experiment state and [From raw events to an A/B decision](../advanced_tutorials/ab/06_End_to_End_Experiment.ipynb)
for a full analysis built on `Dataset`.